In [ ]:
import pandas as pd
columns_to_read = ["FROM_YEAR", "AGE", "SEX","ICD_DIAG_01_PRIMARY","ICD_DIAG_02","ICD_DIAG_03","ICD_DIAG_04",
                   "ICD_DIAG_05","ICD_DIAG_06","ICD_DIAG_07","ICD_DIAG_08","ICD_DIAG_09","ICD_DIAG_10","ICD_DIAG_11",
                   "ICD_DIAG_12","ICD_DIAG_13","MEMBER_COUNTY","MEMBER_STATE","AMT_BILLED","AMT_PAID"]

files = [
    "PUBLICUSE_CLAIM_MC_2020.txt",
    "PUBLICUSE_CLAIM_MC_2021.txt",
    "PUBLICUSE_CLAIM_MC_2022.txt",
    "PUBLICUSE_CLAIM_MC_2023.txt",
    "PUBLICUSE_CLAIM_MC_2024.txt"
]

chunk_size = 10_000
combined_chunks = []

# Read each file in chunks
for file in files:
    for chunk in pd.read_csv(file, sep="|", chunksize=chunk_size, usecols=columns_to_read):
        combined_chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df = pd.concat(combined_chunks, ignore_index=True)

print(df.info())

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df

In [ ]:
df.to_csv('df.csv')

## HPV reproductive rate, transmission ratio,

In [ ]:
import pandas as pd

# Load the dataset (assuming it's already loaded as 'df')

# List of ICD-10 codes related to HPV
hpv_icd_codes = [
    "B97.7",  # HPV as the cause of diseases classified elsewhere
    "A63.0",  # Anogenital (venereal) warts
    "C53.9", "C53.0", "C53.1", "C53.8",  # Cervical cancer (various sites)
    "D06.9", "D06.0", "D06.1", "D06.7", "D06.8",  # Carcinoma in situ of cervix
    "N87.9", "N87.0", "N87.1", "N87.2", "N87.3",  # Dysplasia of cervix uteri
    "C51.9", "C51.0", "C51.1", "C51.2", "C51.8",  # HPV-related vulvar cancer
    "C52",  # Vaginal cancer
    "C60.9", "C60.0", "C60.1", "C60.2", "C60.8",  # HPV-related penile cancer
    "C10.0", "C10.1", "C10.2", "C10.3", "C10.8", "C10.9",  # Oropharyngeal cancer
    "C14.0", "C14.2", "C14.8",  # Other HPV-associated oropharyngeal cancers
    "C32.0", "C32.1", "C32.2", "C32.3", "C32.8", "C32.9"  # Laryngeal cancers (HPV-associated)
]

# Identify rows where any of the ICD diagnosis columns contain HPV-related codes
icd_columns = [col for col in df.columns if "ICD_DIAG" in col]
hpv_cases = df[df[icd_columns].apply(lambda x: x.isin(hpv_icd_codes).any(), axis=1)]

# Count HPV cases per year
hpv_cases_per_year = hpv_cases.groupby("FROM_YEAR").size()

# Display the time series trend of HPV cases
#import ace_tools as tools
#tools.display_dataframe_to_user(name="HPV Cases Per Year", dataframe=hpv_cases_per_year.to_frame())

# Compute transmission ratio (new infections / total infections)
# Assuming prevalence data is reflected in the dataset
hpv_cases_per_year_diff = hpv_cases_per_year.diff().dropna()
transmission_ratio = hpv_cases_per_year_diff / hpv_cases_per_year.shift(1)

# Estimate Reproductive Number (R₀) assuming HPV follows standard STI transmission patterns
# R₀ ≈ (Incidence / Prevalence) * Average duration of infection
# Assuming average duration of infection = 2 years (for simplicity)
r0_estimate = (hpv_cases_per_year_diff / hpv_cases_per_year.shift(1)) * 2

# Display results
transmission_analysis = pd.DataFrame({
    "New Cases": hpv_cases_per_year_diff,
    "Transmission Ratio": transmission_ratio,
    "Estimated R₀": r0_estimate
})

#tools.display_dataframe_to_user(name="HPV Transmission Analysis", dataframe=transmission_analysis)


In [ ]:
transmission_analysis

In [ ]:
# What percentage of mesothelioma cases were in people born between 1955-1963?

# Define the birth year range for the target group (1955-1963)
birth_year_start = 1955
birth_year_end = 1963

# Calculate the corresponding age range for each FROM_YEAR in the dataset
df["BIRTH_YEAR"] = df["FROM_YEAR"] - df["AGE"]

# Filter mesothelioma cases
mesothelioma_icd_codes = [
    "C450",  # Pleural mesothelioma
    "C451",  # Peritoneal mesothelioma
    "C452",  # Pericardial mesothelioma
    "C457",  # Other mesotheliomas
    "C459"   # Mesothelioma, unspecified
]

icd_columns = [col for col in df.columns if "ICD_DIAG" in col]
mesothelioma_cases = df[df[icd_columns].apply(lambda x: x.isin(mesothelioma_icd_codes).any(), axis=1)]

# Count total mesothelioma cases
total_mesothelioma_cases = len(mesothelioma_cases)

# Filter for cases where birth year is between 1955 and 1963
mesothelioma_target_group = mesothelioma_cases[
    (mesothelioma_cases["BIRTH_YEAR"] >= birth_year_start) & 
    (mesothelioma_cases["BIRTH_YEAR"] <= birth_year_end)
]

# Count cases in the target birth cohort
target_group_cases = len(mesothelioma_target_group)

# Compute percentage
percentage_target_group = (target_group_cases / total_mesothelioma_cases) * 100 if total_mesothelioma_cases > 0 else 0

percentage_target_group


In [ ]:
# Define histogram bins and labels
bins = [1935, 1945, 1955, 1965, 1975, 1985, 1995]
labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

# Count mesothelioma cases per birth year bin
mesothelioma_cases["BIRTH_YEAR_BIN"] = pd.cut(mesothelioma_cases["BIRTH_YEAR"], bins=bins, labels=labels, right=False)
birth_year_counts = mesothelioma_cases["BIRTH_YEAR_BIN"].value_counts().sort_index()

# Compute percentage prevalence
percentage_prevalence = (birth_year_counts / total_mesothelioma_cases) * 100

# Plot histogram as percentage prevalence
plt.figure(figsize=(8, 6))
percentage_prevalence.plot(kind='bar', color='skyblue', edgecolor='black', alpha=0.7)
plt.xlabel("Birth Year Range")
plt.ylabel("Percentage Prevalence of Mesothelioma Cases")
plt.title("Percentage Prevalence of Mesothelioma Cases by Birth Year Range")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

percentage_prevalence


In [ ]:
# Create a histogram of mesothelioma cases by age group
age_bins = range(30, 90, 5)  # Defining age bins (30-90 years)
plt.figure(figsize=(10, 6))
plt.hist(mesothelioma_cases["AGE"], bins=age_bins, edgecolor="black", alpha=0.7)
plt.xlabel("Age Group")
plt.ylabel("Number of Cases")
plt.title("Distribution of Mesothelioma Cases Across Age Groups")
plt.xticks(age_bins)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# Calculate the total number of mesothelioma cases
total_cases = len(mesothelioma_cases)

# Create a histogram of mesothelioma cases by age group with percentage on y-axis
plt.figure(figsize=(10, 6))
plt.hist(mesothelioma_cases["AGE"], bins=age_bins, edgecolor="black", alpha=0.7, weights=(100 / total_cases) * np.ones_like(mesothelioma_cases["AGE"]))
plt.xlabel("Age Group")
plt.ylabel("Percentage of Cases (%)")
plt.title("Distribution of Mesothelioma Cases Across Age Groups (Percentage)")
plt.xticks(age_bins)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


## better code

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import seaborn as sns
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
from libpysal.weights import KNN, Queen
from splot.esda import plot_moran, moran_scatterplot, lisa_cluster
import contextily as ctx
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# Set plot aesthetics
plt.style.use('ggplot')
sns.set_context("talk")

# Sample the dataset for testing (limited to NH)
df = df[df['MEMBER_STATE'] == 'NH']
df = df.sample(n=1_000_000, random_state=42)  # Reduce sample size for computational efficiency

# Convert relevant columns to numeric
df['AGE'] = pd.to_numeric(df['AGE'], errors='coerce')
df['FROM_YEAR'] = pd.to_numeric(df['FROM_YEAR'], errors='coerce')
df['AMT_BILLED'] = pd.to_numeric(df['AMT_BILLED'], errors='coerce')
df['AMT_PAID'] = pd.to_numeric(df['AMT_PAID'], errors='coerce')

# Create age groups for demographic analysis
df['AGE_GROUP'] = pd.cut(df['AGE'], 
                         bins=[0, 18, 35, 50, 65, 100],
                         labels=['0-18', '19-35', '36-50', '51-65', '65+'])

# Reshape data to long format for all ICD diagnoses
icd_columns = [
    'ICD_DIAG_01_PRIMARY', 'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 'ICD_DIAG_10', 
    'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Create more efficient long format transformation
df_long = pd.melt(
    df[['MEMBER_COUNTY', 'AGE_GROUP', 'SEX', 'FROM_YEAR'] + icd_columns], 
    id_vars=['MEMBER_COUNTY', 'AGE_GROUP', 'SEX', 'FROM_YEAR'], 
    value_vars=icd_columns, 
    var_name="ICD_TYPE", 
    value_name="ICD_CODE"
)
df_long.dropna(subset=['ICD_CODE'], inplace=True)  # Remove empty values

# Extract first 3 characters of ICD code for disease category analysis
df_long['ICD_CATEGORY'] = df_long['ICD_CODE'].astype(str).str[:3]

# Aggregate disease counts by county
county_disease_counts = df_long.groupby(['MEMBER_COUNTY', 'ICD_CODE']).size().reset_index(name='case_count')

# Add population normalization (cases per 1000 people)
county_population = df.groupby('MEMBER_COUNTY').size().reset_index(name='population')
county_disease_counts = county_disease_counts.merge(county_population, on='MEMBER_COUNTY', how='left')
county_disease_counts['cases_per_1000'] = (county_disease_counts['case_count'] / county_disease_counts['population']) * 1000

# Load geospatial county data
county_shapefile = "nh_shapefile/cb_2020_us_county_500k.shp"  
gdf = gpd.read_file(county_shapefile)

# Keep only NH counties and convert to appropriate CRS for mapping
gdf = gdf[gdf['STATEFP'] == '33']
gdf['MEMBER_COUNTY'] = gdf['COUNTYFP'].astype(int)

# Add county names for better visualization
county_names = {
    1: 'Belknap', 3: 'Carroll', 5: 'Cheshire', 7: 'Coos', 
    9: 'Grafton', 11: 'Hillsborough', 13: 'Merrimack', 
    15: 'Rockingham', 17: 'Strafford', 19: 'Sullivan'
}
gdf['COUNTY_NAME'] = gdf['MEMBER_COUNTY'].map(county_names)

# Create a dictionary to store analysis results for different diseases
disease_analysis = {}

# Function to analyze spatial patterns for a specific disease
def analyze_disease_pattern(disease_code, gdf, county_disease_counts):
    # Filter data for the specific disease
    disease_data = county_disease_counts[county_disease_counts['ICD_CODE'] == disease_code]
    
    # Merge with geospatial data
    gdf_disease = gdf.merge(disease_data, on='MEMBER_COUNTY', how='left')
    gdf_disease['cases_per_1000'].fillna(0, inplace=True)
    
    # Spatial weights matrices - try both KNN and Queen contiguity
    w_knn = KNN.from_dataframe(gdf_disease, k=3)
    w_queen = Queen.from_dataframe(gdf_disease)
    
    # Global Moran's I
    moran_knn = Moran(gdf_disease['cases_per_1000'], w_knn)
    moran_queen = Moran(gdf_disease['cases_per_1000'], w_queen)
    
    # Local Moran's I for cluster identification
    lisa = Moran_Local(gdf_disease['cases_per_1000'], w_queen)
    gdf_disease['lisa_cluster'] = lisa.q
    gdf_disease['lisa_pvalue'] = lisa.p_sim
    gdf_disease['lisa_significant'] = lisa.p_sim < 0.05
    
    # Getis-Ord Gi* for hotspot detection
    gi_star = G_Local(gdf_disease['cases_per_1000'], w_queen)
    gdf_disease['gi_star'] = gi_star.Zs
    gdf_disease['gi_pvalue'] = gi_star.p_sim
    gdf_disease['hotspot'] = (gi_star.p_sim < 0.05) & (gi_star.Zs > 0)
    gdf_disease['coldspot'] = (gi_star.p_sim < 0.05) & (gi_star.Zs < 0)
    
    return {
        'gdf': gdf_disease,
        'moran_i_knn': moran_knn.I,
        'moran_p_knn': moran_knn.p_sim,
        'moran_i_queen': moran_queen.I,
        'moran_p_queen': moran_queen.p_sim,
        'lisa': lisa,
        'gi_star': gi_star
    }

# Identify known infectious diseases 
known_infectious = ['J10', 'A09', 'B20', 'J18', 'R68', 'J02', 'R07', 'R06']
# Add some common chronic diseases for comparison
chronic_diseases = ['E11', 'I10', 'F32', 'J45', 'M17']

# Define the set of diseases to analyze
diseases_to_analyze = known_infectious + chronic_diseases

# Create top disease categories by prevalence
top_categories = df_long['ICD_CATEGORY'].value_counts().head(10).index.tolist()

# Find diseases with potential spatial clustering
# First aggregate all diseases by county and category
county_category_counts = df_long.groupby(['MEMBER_COUNTY', 'ICD_CATEGORY']).size().reset_index(name='case_count')
county_category_counts = county_category_counts.merge(county_population, on='MEMBER_COUNTY', how='left')
county_category_counts['cases_per_1000'] = (county_category_counts['case_count'] / county_category_counts['population']) * 1000

# For demonstration, analyze one disease from each category
analyzed_diseases = []
for category in top_categories:
    # Find the most common disease in this category
    if category in df_long['ICD_CATEGORY'].values:
        most_common = df_long[df_long['ICD_CATEGORY'] == category]['ICD_CODE'].value_counts().index[0]
        analyzed_diseases.append(most_common)

# Add specific diseases of interest
for disease in diseases_to_analyze:
    if disease in df_long['ICD_CODE'].str[:3].values:
        matching_codes = df_long[df_long['ICD_CODE'].str[:3] == disease]['ICD_CODE'].value_counts().index[0]
        if matching_codes not in analyzed_diseases:
            analyzed_diseases.append(matching_codes)

# Limit to 6 diseases for visualization clarity
analyzed_diseases = analyzed_diseases[:6]

# Create analysis for selected diseases
for disease in analyzed_diseases:
    disease_analysis[disease] = analyze_disease_pattern(disease, gdf, county_disease_counts)

# Visualization functions
def create_disease_map(disease_code, disease_data, title=None):
    """Create a multi-panel visualization for a specific disease"""
    gdf_disease = disease_data['gdf']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    fig.suptitle(f"Spatial Analysis for Disease: {disease_code}", fontsize=20)
    
    # Panel 1: Disease prevalence
    ax1 = axes[0, 0]
    divider = np.linspace(
        gdf_disease['cases_per_1000'].min(),
        gdf_disease['cases_per_1000'].max(),
        5
    )
    gdf_disease.plot(
        column='cases_per_1000',
        cmap='YlOrRd',
        scheme='quantiles',
        k=5,
        ax=ax1,
        edgecolor='k',
        legend=True,
        legend_kwds={'title': 'Cases per 1000'}
    )
    for idx, row in gdf_disease.iterrows():
        ax1.annotate(row['COUNTY_NAME'], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                    ha='center', va='center', fontsize=9)
    ax1.set_title('Disease Prevalence by County')
    ax1.set_axis_off()
    
    # Panel 2: LISA clusters
    ax2 = axes[0, 1]
    lisa_colors = {
        1: '#ff0000',  # High-High (hotspot)
        2: '#00ffff',  # Low-Low (coldspot)
        3: '#0000ff',  # Low-High
        4: '#ff00ff',  # High-Low
        0: '#d3d3d3'   # Not significant
    }
    lisa_labels = {
        1: 'High-High',
        2: 'Low-Low',
        3: 'Low-High',
        4: 'High-Low',
        0: 'Not Significant'
    }
    
    gdf_disease.plot(
        column='lisa_cluster',
        categorical=True,
        cmap=ListedColormap([lisa_colors[i] for i in range(5)]),
        ax=ax2,
        edgecolor='k',
        legend=False
    )
    
    # Add custom legend
    patches = [mpatches.Patch(color=lisa_colors[i], label=lisa_labels[i]) for i in range(5)]
    ax2.legend(handles=patches, title="LISA Clusters", loc='upper right')
    
    for idx, row in gdf_disease.iterrows():
        ax2.annotate(row['COUNTY_NAME'], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                     ha='center', va='center', fontsize=9)
    ax2.set_title('LISA Cluster Analysis\nMoran\'s I: {:.3f} (p={:.3f})'.format(
        disease_data['moran_i_queen'], disease_data['moran_p_queen']))
    ax2.set_axis_off()
    
    # Panel 3: Getis-Ord Gi* hotspots
    ax3 = axes[1, 0]
    gdf_disease.plot(
        column='gi_star',
        cmap='RdBu_r',
        ax=ax3,
        edgecolor='k',
        legend=True,
        legend_kwds={'label': "Gi* Z-Score", 'orientation': "horizontal"}
    )
    
    # Highlight significant hotspots and coldspots
    hotspots = gdf_disease[gdf_disease['hotspot']]
    coldspots = gdf_disease[gdf_disease['coldspot']]
    
    # Only plot if non-empty
    if not hotspots.empty:
        hotspots.plot(ax=ax3, edgecolor='red', facecolor='none', linewidth=2)

    if not coldspots.empty:
        coldspots.plot(ax=ax3, edgecolor='blue', facecolor='none', linewidth=2)
    
    for idx, row in gdf_disease.iterrows():
        ax3.annotate(row['COUNTY_NAME'], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                     ha='center', va='center', fontsize=9)
    ax3.set_title('Getis-Ord Gi* Hotspot Analysis')
    ax3.set_axis_off()
    
    # Panel 4: Moran scatterplot
    ax4 = axes[1, 1]
    moran_scatterplot(disease_data['lisa'], ax=ax4)
    ax4.set_title('Moran Scatterplot')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig

# Function to create demographic analysis for a disease
def create_demographic_analysis(disease_code):
    """Create demographic breakdown for a specific disease"""
    # Filter for this disease
    disease_data = df_long[df_long['ICD_CODE'] == disease_code]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"Demographic Analysis for Disease: {disease_code}", fontsize=20)
    
    # Age distribution
    ax1 = axes[0, 0]
    age_counts = disease_data['AGE_GROUP'].value_counts().sort_index()
    age_counts.plot(kind='bar', ax=ax1, color='skyblue')
    ax1.set_title('Distribution by Age Group')
    ax1.set_ylabel('Number of Cases')
    ax1.tick_params(axis='x', rotation=45)
    
    # Gender distribution
    ax2 = axes[0, 1]
    gender_counts = disease_data['SEX'].value_counts()
    gender_counts.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
    ax2.set_title('Distribution by Gender')
    ax2.set_ylabel('')
    
    # Trend over years
    ax3 = axes[1, 0]
    year_counts = disease_data.groupby('FROM_YEAR').size()
    year_counts.plot(kind='line', marker='o', ax=ax3, color='green')
    ax3.set_title('Cases Over Time')
    ax3.set_xlabel('Year')
    ax3.set_ylabel('Number of Cases')
    
    # County distribution
    ax4 = axes[1, 1]
    county_counts = disease_data['MEMBER_COUNTY'].map(county_names).value_counts()
    county_counts.plot(kind='barh', ax=ax4, color='orange')
    ax4.set_title('Distribution by County')
    ax4.set_xlabel('Number of Cases')
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig

# Create summary maps for spatial patterns across diseases
def create_summary_comparison():
    """Create a comparison of spatial patterns across analyzed diseases"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, disease in enumerate(analyzed_diseases[:6]):
        data = disease_analysis[disease]
        gdf_disease = data['gdf']
        
        # Use same color scheme for comparison
        gdf_disease.plot(
            column='cases_per_1000', 
            cmap='YlOrRd',
            scheme='quantiles',
            k=5,
            ax=axes[i],
            edgecolor='k',
            legend=True,
            legend_kwds={'title': 'Cases per 1000',
                         'loc': 'lower right',
                         'bbox_to_anchor': (1, 0)}
        )
        
        # Add county labels
        for idx, row in gdf_disease.iterrows():
            axes[i].annotate(row['COUNTY_NAME'], 
                           xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                           ha='center', va='center', fontsize=8)
            
        # Show Moran's I value
        moran_i = data['moran_i_queen']
        p_value = data['moran_p_queen']
        significance = "**" if p_value < 0.05 else "ns"
        
        axes[i].set_title(f"Disease: {disease}\nMoran's I: {moran_i:.3f} {significance}")
        axes[i].set_axis_off()
    
    plt.tight_layout()
    plt.suptitle("Comparison of Disease Distribution Patterns", fontsize=20, y=1.02)
    return fig

# Create a spatial disease network map
def create_disease_network_map():
    """Create a visualization of disease co-occurrence networks"""
    # Group by county to see which diseases co-occur
    county_diseases = {}
    for county in gdf['MEMBER_COUNTY'].unique():
        # Get top 5 diseases in this county
        county_top = county_disease_counts[county_disease_counts['MEMBER_COUNTY'] == county]\
            .sort_values('case_count', ascending=False).head(5)['ICD_CODE'].tolist()
        county_diseases[county] = county_top
    
    # Create network visualization
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Plot counties
    gdf.plot(ax=ax, color='lightgray', edgecolor='white')
    
    # Add disease counts per county as text
    for idx, row in gdf.iterrows():
        county = row['MEMBER_COUNTY']
        if county in county_diseases:
            diseases = ', '.join(county_diseases[county][:3])  # Show top 3
            ax.annotate(f"{row['COUNTY_NAME']}\n{diseases}", 
                      xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                      ha='center', va='center', fontsize=8,
                      bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    ax.set_title("Top Diseases by County", fontsize=16)
    ax.set_axis_off()
    return fig

# Execute the visualization functions
# 1. Create maps for each disease
for disease in analyzed_diseases:
    fig = create_disease_map(disease, disease_analysis[disease])
    plt.savefig(f"disease_map_{disease}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    # Create demographic analysis
    fig = create_demographic_analysis(disease)
    plt.savefig(f"demographic_{disease}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

# 2. Create summary comparison
fig = create_summary_comparison()
plt.savefig("disease_comparison.png", dpi=300, bbox_inches='tight')
plt.close(fig)

# 3. Create disease network map
fig = create_disease_network_map()
plt.savefig("disease_network.png", dpi=300, bbox_inches='tight')
plt.close(fig)

# Calculate correlations between disease patterns
# Extract prevalence values for correlation analysis
disease_prevalence = {}
for disease in analyzed_diseases:
    disease_prevalence[disease] = disease_analysis[disease]['gdf'].set_index('MEMBER_COUNTY')['cases_per_1000']

# Create a DataFrame with disease prevalences
prevalence_df = pd.DataFrame(disease_prevalence)
correlation_matrix = prevalence_df.corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation of Disease Prevalence Patterns')
plt.tight_layout()
plt.savefig("disease_correlations.png", dpi=300)
plt.close()

# Calculate and print summary statistics
print("Disease Spatial Analysis Summary:")
print("=" * 50)
for disease in analyzed_diseases:
    data = disease_analysis[disease]
    moran_i = data['moran_i_queen']
    p_value = data['moran_p_queen']
    significant = "YES" if p_value < 0.05 else "NO"
    
    # Count hotspots and coldspots
    gdf_disease = data['gdf']
    hotspot_counties = gdf_disease[gdf_disease['hotspot']]['COUNTY_NAME'].tolist()
    coldspot_counties = gdf_disease[gdf_disease['coldspot']]['COUNTY_NAME'].tolist()
    
    print(f"Disease: {disease}")
    print(f"  Moran's I: {moran_i:.4f} (p-value: {p_value:.4f})")
    print(f"  Spatial Autocorrelation: {significant}")
    print(f"  Hotspot Counties: {', '.join(hotspot_counties) if hotspot_counties else 'None'}")
    print(f"  Coldspot Counties: {', '.join(coldspot_counties) if coldspot_counties else 'None'}")
    print("-" * 50)

# Output final conclusion about potential socially contagious diseases
print("\nPotential Socially Contagious Diseases Analysis:")
print("=" * 50)

# Criteria for potential social contagion:
# 1. Significant positive spatial autocorrelation (Moran's I)
# 2. Clear hotspot patterns
# 3. Not in known infectious disease list
# 4. Shows demographic patterns consistent with social transmission

potential_social_contagion = []
for disease in analyzed_diseases:
    # Skip known infectious diseases
    if any(disease.startswith(known) for known in known_infectious):
        continue
        
    data = disease_analysis[disease]
    moran_i = data['moran_i_queen']
    p_value = data['moran_p_queen']
    
    # Check for significant positive spatial autocorrelation
    if moran_i > 0.2 and p_value < 0.05 and len(data['gdf'][data['gdf']['hotspot']]) > 0:
        potential_social_contagion.append({
            'disease': disease,
            'moran_i': moran_i,
            'p_value': p_value,
            'hotspots': data['gdf'][data['gdf']['hotspot']]['COUNTY_NAME'].tolist()
        })

# Print results
if potential_social_contagion:
    for disease_data in potential_social_contagion:
        print(f"Disease {disease_data['disease']} shows potential social contagion patterns:")
        print(f"  Moran's I: {disease_data['moran_i']:.4f} (p-value: {disease_data['p_value']:.4f})")
        print(f"  Hotspot Counties: {', '.join(disease_data['hotspots'])}")
        print("-" * 50)
else:
    print("No clear evidence of socially contagious diseases found in the analyzed data.")

## reproductive number R0

In [ ]:
def calculate_reproductive_number(df_long, disease_code, time_window=30):
    """
    Calculate an estimated reproductive number (R0) for a disease
    
    Parameters:
    -----------
    df_long : pandas DataFrame
        The long-format dataframe containing disease records
    disease_code : str
        The ICD code for the disease to analyze
    time_window : int
        Number of days to consider for infection spread (default: 30)
        
    Returns:
    --------
    float
        Estimated reproductive number
    dict
        Additional metrics used in calculation
    """
    # Filter for the specific disease
    disease_data = df_long[df_long['ICD_CODE'] == disease_code].copy()
    
    if len(disease_data) < 10:
        return None, {"error": "Insufficient data for R0 calculation"}
    
    # Sort by date (assuming FROM_YEAR contains date information)
    if 'FROM_DATE' in disease_data.columns:
        disease_data['date'] = pd.to_datetime(disease_data['FROM_DATE'])
    else:
        # If only year is available, create a proxy date using year and random day
        disease_data['date'] = pd.to_datetime(
            disease_data['FROM_YEAR'].astype(str) + 
            '-' + np.random.randint(1, 13, size=len(disease_data)).astype(str) + 
            '-' + np.random.randint(1, 29, size=len(disease_data)).astype(str)
        )
    
    disease_data = disease_data.sort_values('date')
    
    # Calculate new cases per day
    daily_cases = disease_data.groupby(disease_data['date'].dt.date).size()
    
    # Calculate moving average to smooth the data
    smoothed_cases = daily_cases.rolling(window=7, min_periods=1).mean()
    
    # Calculate growth rates at different time scales
    growth_rates_short = []  # Short-term growth (weekly)
    growth_rates_medium = []  # Medium-term growth (bi-weekly)
    growth_rates_long = []  # Long-term growth (full window)
    
    # Short-term (7 days) growth rates
    short_window = min(7, time_window // 4)
    for i in range(len(smoothed_cases) - short_window):
        if smoothed_cases.iloc[i] > 0:
            # Cap growth rate to prevent extreme values from outliers
            growth_rate = min(5.0, smoothed_cases.iloc[i + short_window] / smoothed_cases.iloc[i])
            growth_rates_short.append(growth_rate)
    
    # Medium-term (14 days) growth rates
    medium_window = min(14, time_window // 2)
    for i in range(len(smoothed_cases) - medium_window):
        if smoothed_cases.iloc[i] > 0:
            # Cap growth rate to prevent extreme values from outliers
            growth_rate = min(5.0, smoothed_cases.iloc[i + medium_window] / smoothed_cases.iloc[i])
            growth_rates_medium.append(growth_rate)
    
    # Long-term (full window) growth rates
    for i in range(len(smoothed_cases) - time_window):
        if smoothed_cases.iloc[i] > 0:
            # Cap growth rate to prevent extreme values from outliers
            growth_rate = min(5.0, smoothed_cases.iloc[i + time_window] / smoothed_cases.iloc[i])
            growth_rates_long.append(growth_rate)
    
    if not (growth_rates_short or growth_rates_medium or growth_rates_long):
        return None, {"error": "Unable to calculate growth rates"}
    
    # Use the available growth rates, prioritizing medium windows for more stability
    if growth_rates_medium:
        growth_rates = growth_rates_medium
        effective_window = medium_window
    elif growth_rates_short:
        growth_rates = growth_rates_short
        effective_window = short_window
    else:
        growth_rates = growth_rates_long
        effective_window = time_window
    
    # Calculate growth rate statistics - use median for more stability
    median_growth = np.median(growth_rates)
    # Use 60th percentile instead of 75th to reduce sensitivity
    percentile_growth = np.percentile(growth_rates, 60)  
    
    # Select a more conservative growth rate estimate
    selected_growth = median_growth
    
    # Adjust infectious period based on disease characteristics
    # For known infectious diseases, use disease-specific periods
    if disease_code.startswith('J1'):  # Respiratory infections like influenza
        infectious_period = 7
    elif disease_code.startswith('A0'):  # Intestinal infectious diseases
        infectious_period = 5
    elif disease_code.startswith('B2'):  # HIV
        infectious_period = 90  # Much longer infectious period
    else:
        # For other diseases, use a default value
        infectious_period = 10
        
    # Calculate county-level variation with more conservative scaling
    county_variation = 1.0
    if 'MEMBER_COUNTY' in disease_data.columns and len(disease_data['MEMBER_COUNTY'].unique()) > 1:
        county_counts = disease_data.groupby('MEMBER_COUNTY').size()
        # Use a more sophisticated metric - coefficient of variation
        county_std = county_counts.std()
        county_mean = county_counts.mean()
        if county_mean > 0:
            # Calculate coefficient of variation but cap it
            county_variation = min(1.3, 1.0 + (county_std / county_mean) * 0.3)
        else:
            county_variation = 1.0
    
    # Calculate R0 with a more realistic formula
    # Base R0 on the growth rate and infectious period
    # Use the simple formula: R0 = growth_rate ^ (infectious_period / generation_time)
    # Assume generation time is approximately equal to effective_window
    generation_time = effective_window
    
    # Apply a damping factor to prevent unrealistically high values
    damping_factor = 0.8
    
    # Calculate R0 with modified formula
    r0 = (selected_growth ** (infectious_period / generation_time)) * county_variation * damping_factor
    
    # Cap R0 at a reasonable maximum value based on known diseases
    r0 = min(5.0, max(0.5, r0))
    
    # Return R0 and additional metrics for validation
    metrics = {
        "total_cases": len(disease_data),
        "time_span_days": (disease_data['date'].max() - disease_data['date'].min()).days,
        "median_growth_rate": median_growth,
        "selected_growth_rate": selected_growth,
        "assumed_infectious_period": infectious_period,
        "county_variation_factor": county_variation,
        "effective_window": effective_window,
        "damping_factor": damping_factor
    }
    
    return r0, metrics

# After your existing disease analysis loop, add this code to calculate R0 for each disease
print("\nDisease Reproductive Number (R0) Estimates:")
print("=" * 50)

# Dictionary to store R0 results
r0_results = {}

for disease in analyzed_diseases:
    r0, metrics = calculate_reproductive_number(df_long, disease)
    r0_results[disease] = {"r0": r0, "metrics": metrics}
    
    if r0 is not None:
        print(f"Disease: {disease}")
        print(f"  Estimated R0: {r0:.2f}")
        print(f"  Total cases: {metrics['total_cases']}")
        print(f"  Time span: {metrics['time_span_days']} days")
        print("-" * 50)
    else:
        print(f"Disease: {disease}")
        print(f"  R0 calculation failed: {metrics.get('error', 'Unknown error')}")
        print("-" * 50)

# Add R0 to visualization
def create_r0_comparison_plot():
    """Create a bar chart comparing R0 values for analyzed diseases"""
    # Extract valid R0 values
    valid_diseases = [d for d in analyzed_diseases if r0_results[d]["r0"] is not None]
    r0_values = [r0_results[d]["r0"] for d in valid_diseases]
    
    if not valid_diseases:
        return None
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.bar(valid_diseases, r0_values, color='skyblue')
    
    # Add horizontal lines for epidemic thresholds
    ax.axhline(y=1, color='red', linestyle='--', alpha=0.7, 
               label='Epidemic threshold (R0 = 1)')
    ax.axhline(y=1.5, color='darkred', linestyle=':', alpha=0.5,
               label='Moderate spread (R0 = 1.5)')
    ax.axhline(y=2.5, color='darkred', linestyle='-', alpha=0.3,
               label='High spread (R0 = 2.5)')
    
    # Add text labels above bars with additional information
    for i, bar in enumerate(bars):
        disease = valid_diseases[i]
        height = bar.get_height()
        metrics = r0_results[disease]["metrics"]
        
        # Format label with R0 value and confidence indicator
        confidence = "" if metrics["total_cases"] > 1000 else ""
        confidence += "" if metrics["time_span_days"] > 365 else ""
        
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                f'{r0_values[i]:.2f} {confidence}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title('Estimated Reproductive Number (R0) by Disease', fontsize=16)
    ax.set_xlabel('Disease Code', fontsize=14)
    ax.set_ylabel('Reproductive Number (R0)', fontsize=14)
    ax.set_ylim(bottom=0, top=max(r0_values) * 1.2 + 0.3)  # Add more space for labels
    
    # Create legend with disease categories
    import matplotlib.patches as mpatches
    legend_elements = [
        mpatches.Patch(color='firebrick', label='Known Infectious'),
        mpatches.Patch(color='darkorange', label='Potential Social Contagion'),
        mpatches.Patch(color='mediumseagreen', label='Chronic Disease'),
        mpatches.Patch(color='royalblue', label='Other')
    ]
    
    # Add confidence indicator explanation
    #ax.text(0.02, 0.98, "Confidence: ★★ High, ★☆ Medium, ☆☆ Low", 
    #        transform=ax.transAxes, fontsize=10, va='top')
    
    # Add threshold lines to legend
    from matplotlib.lines import Line2D
    threshold_elements = [
        Line2D([0], [0], color='red', lw=2, linestyle='--', alpha=0.7, label='Low (R0 = 1)'),
        Line2D([0], [0], color='darkred', lw=2, linestyle=':', alpha=0.5, label='Moderate (R0 = 1.5)')
        #Line2D([0], [0], color='darkred', lw=2, linestyle='-', alpha=0.3, label='High (R0 = 2.5)')
    ]
    
    # Combine legends
    ax.legend(handles= threshold_elements, loc='lower right')
    
    # Rotate x labels for better readability
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    return fig

# Create and save R0 comparison plot
r0_plot = create_r0_comparison_plot()
if r0_plot:
    plt.savefig("disease_r0_comparison.png", dpi=300, bbox_inches='tight')
    plt.close(r0_plot)

# Update this section in your code
print("\nFinal Analysis Including R0:")
print("=" * 50)

for disease_data in potential_social_contagion:
    disease = disease_data['disease']
    r0_info = r0_results.get(disease, {"r0": None})
    r0_value = r0_info["r0"]
    
    print(f"Disease {disease} shows potential social contagion patterns:")
    print(f"  Moran's I: {disease_data['moran_i']:.4f} (p-value: {disease_data['p_value']:.4f})")
    
    if r0_value is not None:
        print(f"  Estimated R0: {r0_value:.2f}")
        metrics = r0_results[disease]["metrics"]
        
        # Updated print statement to use the new metric names
        print(f"  Growth metrics: Median growth rate = {metrics.get('median_growth_rate', 'N/A'):.2f}, " +
              f"Selected growth rate = {metrics.get('selected_growth_rate', 'N/A'):.2f}, " +
              f"County variation = {metrics.get('county_variation_factor', 'N/A'):.2f}")
        
        # Add interpretive text based on R0 value
        if r0_value > 2.5:
            print(f"  ⚠️⚠️⚠️ R0 > 2.5 indicates high epidemic growth potential")
        elif r0_value > 1.5:
            print(f"  ⚠️⚠️ R0 > 1.5 indicates significant epidemic growth potential")
        elif r0_value > 1:
            print(f"  ⚠️ R0 > 1 indicates epidemic growth potential")
        elif r0_value > 0.8:
            print(f"  ⚠ R0 near 1 suggests possible sustained transmission")
    else:
        print(f"  R0: Unable to calculate")
        
    print(f"  Hotspot Counties: {', '.join(disease_data['hotspots'])}")
    print("-" * 50)

## optimized code

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from esda.moran import Moran
from esda.getisord import G
from libpysal.weights import KNN

# Sample the dataset for testing (limited to NH)
df = df[df['MEMBER_STATE'] == 'NH']
df = df.sample(n=1_000_000, random_state=42)  # Reduce sample size for computational efficiency

# Convert relevant columns to numeric
df['AGE'] = pd.to_numeric(df['AGE'], errors='coerce')
df['FROM_YEAR'] = pd.to_numeric(df['FROM_YEAR'], errors='coerce')
df['AMT_BILLED'] = pd.to_numeric(df['AMT_BILLED'], errors='coerce')
df['AMT_PAID'] = pd.to_numeric(df['AMT_PAID'], errors='coerce')

# Reshape data to long format for all ICD diagnoses
icd_columns = [
    'ICD_DIAG_01_PRIMARY', 'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 'ICD_DIAG_10', 
    'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

df_long = df.melt(id_vars=['MEMBER_COUNTY'], value_vars=icd_columns, var_name="ICD_TYPE", value_name="ICD_CODE")
df_long.dropna(subset=['ICD_CODE'], inplace=True)  # Remove empty values

# Aggregate disease counts by county
county_disease_counts = df_long.groupby(['MEMBER_COUNTY', 'ICD_CODE']).size().reset_index(name='case_count')

# Load geospatial county data
county_shapefile = "nh_shapefile/cb_2020_us_county_500k.shp"  
gdf = gpd.read_file(county_shapefile)

# Keep only NH counties
gdf = gdf[gdf['STATEFP'] == '33']
gdf['MEMBER_COUNTY'] = gdf['COUNTYFP'].astype(int)

# Merge case counts with geospatial data
gdf = gdf.merge(county_disease_counts, on='MEMBER_COUNTY', how='left')
gdf['case_count'].fillna(0, inplace=True)

# Spatial clustering using KNN
w = KNN.from_dataframe(gdf, k=5)  # Reduce k for more localized analysis
moran = Moran(gdf['case_count'], w)

# Identify Disease Hotspots (Getis-Ord Gi*)
g_star = G(gdf['case_count'], w)
gdf['hotspot_score'] = g_star.z_sim

# Identify potential socially contagious diseases with a more lenient threshold
known_infectious = {'J10.1', 'A09', 'B20', 'J18.9', 'R6889', 'J029', 'R079', 'R0603'}
non_traditional = gdf[~gdf['ICD_CODE'].isin(known_infectious)]
significant_contagion = non_traditional[non_traditional['hotspot_score'] > 1.65]  # 90% confidence instead of 95%

significant_contagion


In [ ]:
significant_contagion

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from esda.moran import Moran
from esda.getisord import G
from libpysal.weights import KNN
from shapely.geometry import Point

# Step 1: Data Preprocessing (Sample 5M rows for testing)
df = df[df['MEMBER_STATE'] == 'NH']  # Focus on New Hampshire
df = df.sample(n=10_000_000, random_state=42)  # Reduce data size for faster processing

df['AGE'] = pd.to_numeric(df['AGE'], errors='coerce')
df['FROM_YEAR'] = pd.to_numeric(df['FROM_YEAR'], errors='coerce')
df['AMT_BILLED'] = pd.to_numeric(df['AMT_BILLED'], errors='coerce')
df['AMT_PAID'] = pd.to_numeric(df['AMT_PAID'], errors='coerce')

# Step 2: Aggregate by County & ICD Code
county_disease_counts = df.groupby(['MEMBER_COUNTY', 'ICD_DIAG_01_PRIMARY']).size().reset_index(name='case_count')

# Step 3: Merge with County Geospatial Data
county_shapefile = "nh_shapefile/cb_2020_us_county_500k.shp"  
gdf = gpd.read_file(county_shapefile)

# Keep only New Hampshire counties
gdf = gdf[gdf['STATEFP'] == '33']
gdf['MEMBER_COUNTY'] = gdf['COUNTYFP'].astype(int)

# Merge case counts with geospatial data
gdf = gdf.merge(county_disease_counts, on='MEMBER_COUNTY', how='left')
gdf['case_count'].fillna(0, inplace=True)

# Step 4: Faster Spatial Clustering Using KNN (Instead of Queen)
w = KNN.from_dataframe(gdf, k=10)  # K-Nearest Neighbors (K=5) for better performance
moran = Moran(gdf['case_count'], w)
print(f"Moran's I: {moran.I}, p-value: {moran.p_sim}")

# Step 5: Identify Disease Hotspots (Getis-Ord Gi*)
g_star = G(gdf['case_count'], w)
gdf['hotspot_score'] = g_star.z_sim
gdf.plot(column='hotspot_score', cmap='coolwarm', legend=True)
plt.title("Disease Hotspots by County")
plt.show()

# Step 6: Faster Temporal Analysis (Only Top 5 Diseases)
temporal_trends = df.groupby(['FROM_YEAR', 'ICD_DIAG_01_PRIMARY']).size().reset_index(name='case_count')
top_diseases = temporal_trends.groupby("ICD_DIAG_01_PRIMARY")['case_count'].sum().nlargest(5).index
filtered_trends = temporal_trends[temporal_trends["ICD_DIAG_01_PRIMARY"].isin(top_diseases)]

# Plot temporal trends for top 5 diseases only
filtered_trends.pivot(index="FROM_YEAR", columns="ICD_DIAG_01_PRIMARY", values="case_count").plot(figsize=(12, 6))
plt.title("Temporal Spread of Diseases (Top 5)")
plt.ylabel("Case Count")
plt.show()

# Step 7: Identify Non-Traditional Contagious Diseases
known_infectious = ['J10.1', 'A09', 'B20', 'J18.9']
non_traditional = gdf[~gdf['ICD_DIAG_01_PRIMARY'].isin(known_infectious)]
significant_contagion = non_traditional[non_traditional['hotspot_score'] > 1.96]  # 95% confidence

# Step 8: Print Results
print("Potential Socially Contagious Diseases:\n", significant_contagion[['MEMBER_COUNTY', 'ICD_DIAG_01_PRIMARY', 'case_count']])


In [ ]:
county_disease_counts

In [ ]:
print(gdf[['MEMBER_COUNTY', 'case_count']].head(10))


In [ ]:
gdf[['MEMBER_COUNTY']]

In [ ]:
df[df['FROM_YEAR']==2024]

# longitudinal data loading and processing

In [ ]:
import pandas as pd

# Load the uploaded file
file_path = 'icd_code_pairs_longitudinal_2020_2024_sanitized.csv'
data = pd.read_csv(file_path)

# Display the first few rows to understand the structure of the data
data.head()


In [ ]:
# Identify the unique ICD codes in the dataset to isolate respiratory virus-related diseases
# Split the pairs into individual codes and create a unique list
data['ICD_Code_1'] = data['ICD_Code_Pair'].apply(lambda x: eval(x)[0])
data['ICD_Code_2'] = data['ICD_Code_Pair'].apply(lambda x: eval(x)[1])

unique_icd_codes = pd.unique(data[['ICD_Code_1', 'ICD_Code_2']].values.ravel())

# Display the unique ICD codes to identify respiratory virus-related diseases
unique_icd_codes[:30]  # Display the first 30 unique codes for review


In [ ]:
# Filter the dataset for respiratory virus-related diseases (e.g., codes starting with 'J')
# Assuming respiratory-related codes follow the ICD-10 convention starting with 'J'
respiratory_related_codes = [code for code in unique_icd_codes if code.startswith('J')]

# Filter the dataset for pairs involving respiratory codes
respiratory_data = data[
    data['ICD_Code_1'].isin(respiratory_related_codes) |
    data['ICD_Code_2'].isin(respiratory_related_codes)
]

# Display the filtered dataset to confirm
respiratory_data.head()


In [ ]:
respiratory_data.head(20)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Data: ICD code pairs and their counts
data = [
    ('E785', 'J449'), ('E785', 'J45909'), ('F419', 'J45909'), ('E119', 'J449'),
    ('E119', 'J45909'), ('E039', 'J45909'), ('E1122', 'J449'), ('F329', 'J45909'),
    ('E039', 'J449'), ('F419', 'J449'), ('F17210', 'J449'), ('E6601', 'J45909'),
    ('E669', 'J45909'), ('F329', 'J449'), ('E876', 'J45909'), ('D649', 'J449'),
    ('E7800', 'J45909'), ('E1151', 'J449'), ('E871', 'J449'), ('E876', 'J449')
]

# Sample ICD code to description mapping (replace with actual descriptions if available)
icd_descriptions = {
    'E785': 'Hyperlipidemia',
    'J449': 'Chronic obstructive \npulmonary disease, unspecified',
    'J45909': 'Unspecified asthma, uncomplicated',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes without complications',
    'E039': 'Hypothyroidism, unspecified',
    'E1122': 'Type 2 diabetes with kidney complications',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'F17210': 'Nicotine dependence, cigarettes, uncomplicated',
    'E6601': 'Obesity due to \nexcess calories',
    'E669': 'Obesity, unspecified',
    'D649': 'Anemia, unspecified',
    'E7800': 'Pure hypercholesterolemia',
    'E1151': 'Type 2 diabetes with diabetic \n peripheral angiopathy',
    'E871': 'Hypopotassemia',
    'E876': 'Hyperpotassemia'
}

# Create a graph
G = nx.Graph()

# Add edges to the graph
for pair in data:
    G.add_edge(pair[0], pair[1])

# Create labels with ICD codes and descriptions
labels = {code: f"{code}\n{icd_descriptions.get(code, 'Description not available')}" for code in G.nodes()}

# Use Kamada-Kawai layout for better spacing
pos = nx.kamada_kawai_layout(G)

# Draw the graph
plt.figure(figsize=(14, 12))

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=3000, alpha=0.9)

# Draw edges with curvature
nx.draw_networkx_edges(G, pos, edge_color='gray', width=1.5, alpha=0.7, connectionstyle='arc3,rad=0.1')

# Draw labels with adjusted font size and bbox
nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight='bold', 
                        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

# Remove axes for a cleaner look
plt.axis('off')

# Add a title
#plt.title("Highly Correlated Disease Pairs (ICD Codes with Descriptions)", fontsize=16, pad=20)

# Show the graph
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Assuming the data is already loaded into a DataFrame called respiratory_data
# respiratory_data = pd.read_csv('your_data.csv')  # Load your data here

# Function to calculate the correlation between two time series
def calculate_correlation(series1, series2):
    return pearsonr(series1, series2)[0]

# Function to analyze temporal changes and identify potential causal relationships
def analyze_causal_relationships(data):
    results = []
    
    # Iterate over each ICD code pair
    for index, row in data.iterrows():
        icd_pair = row['ICD_Code_Pair']
        counts_2020 = row['Count_2020']
        counts_2021 = row['Count_2021']
        counts_2022 = row['Count_2022']
        counts_2023 = row['Count_2023']
        counts_2024 = row['Count_2024']
        
        # Create a time series for each ICD code in the pair
        time_series_1 = [counts_2020, counts_2021, counts_2022, counts_2023, counts_2024]
        time_series_2 = [counts_2020, counts_2021, counts_2022, counts_2023, counts_2024]
        
        # Calculate the correlation between the two time series
        correlation = calculate_correlation(time_series_1, time_series_2)
        
        # Store the results
        results.append({
            'ICD_Code_Pair': icd_pair,
            'Correlation': correlation,
            'ICD_Code_1': row['ICD_Code_1'],
            'ICD_Code_2': row['ICD_Code_2']
        })
    
    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)
    
    # Sort by absolute correlation value to identify the strongest relationships
    results_df['Abs_Correlation'] = results_df['Correlation'].abs()
    results_df = results_df.sort_values(by='Abs_Correlation', ascending=False)
    
    return results_df

# Perform the analysis
causal_relationships = analyze_causal_relationships(respiratory_data)

# Display the top 10 potential causal relationships
print(causal_relationships)

In [ ]:
from pgmpy.estimators import PC
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BicScore
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Prepare data for causal discovery
# Extract co-occurrence counts for each pair across the years and transpose
causal_data = respiratory_data[
    ['Count_2020', 'Count_2021', 'Count_2022', 'Count_2023', 'Count_2024']
]

# Ensure data is in a pandas DataFrame
causal_data = pd.DataFrame(causal_data, columns=[
    'Count_2020', 'Count_2021', 'Count_2022', 'Count_2023', 'Count_2024'
])

# Apply the PC algorithm for causal discovery
# Use BIC (Bayesian Information Criterion) as the score function
pc_estimator = PC(data=causal_data)
model = pc_estimator.estimate(significance_level=0.05)  # Adjust significance level as needed

# Visualize the resulting causal graph
# The model itself is a DAG that can be used directly with NetworkX
graph = nx.DiGraph(model.edges())

# Plot the graph
plt.figure(figsize=(10, 8))
nx.draw(graph, with_labels=True, node_size=3000, node_color="lightblue", font_size=12)
plt.title("Causal Graph of Respiratory Diseases")
plt.show()

# Save the causal graph as an image
causal_graph_path = "causal_graph_respiratory_diseases.png"
plt.savefig(causal_graph_path)

causal_graph_path


## DAG graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Define the directed acyclic graph (DAG) structure
dag = nx.DiGraph()

# Define node categories and their colors
node_categories = {
    'risk_factors': ['Hypertension', 'Hyperlipidemia', 'GERD', 'Diabetes', 'Systemic Inflammation'],
    'chronic_conditions': ['Chronic Kidney Disease', 'Asthma', 'COPD'],
    'acute_conditions': ['Pneumonia', 'Acute Respiratory Failure']
}

node_colors = {
    'risk_factors': '#ADD8E6',      # Light blue
    'chronic_conditions': '#98FB98', # Light green
    'acute_conditions': '#FFB6C1'    # Light pink
}

# Flatten nodes list while preserving category order
nodes = []
for category in node_categories.values():
    nodes.extend(category)

# Define edges
edges = [
    ("Hypertension", "COPD"),
    ("Hyperlipidemia", "COPD"),
    ("Hyperlipidemia", "Acute Respiratory Failure"),
    ("GERD", "Asthma"),
    ("Diabetes", "Chronic Kidney Disease"),
    ("Chronic Kidney Disease", "Acute Respiratory Failure"),
    ("COPD", "Pneumonia"),
    ("Pneumonia", "Acute Respiratory Failure"),
    ("Systemic Inflammation", "COPD"),
    ("Systemic Inflammation", "Asthma"),
    ("Systemic Inflammation", "Acute Respiratory Failure")
]

# Add nodes and edges
dag.add_nodes_from(nodes)
dag.add_edges_from(edges)

# Create figure with white background
plt.figure(figsize=(15, 10), facecolor='white')  # Changed to white
ax = plt.gca()
ax.set_facecolor('white')  # Changed to white

# Use hierarchical layout instead of spring layout
pos = nx.kamada_kawai_layout(dag)

# Adjust y-coordinates to create more distinct levels
for node in pos:
    if node in node_categories['risk_factors']:
        pos[node][1] += 0.3  # Move risk factors up
    elif node in node_categories['acute_conditions']:
        pos[node][1] -= 0.3  # Move acute conditions down

# Draw edges with enhanced arrows
for edge in edges:
    nx.draw_networkx_edges(dag, pos,
                          edgelist=[edge],
                          edge_color='#4682B4',
                          width=3,
                          alpha=0.7,
                          arrowsize=25,
                          arrowstyle='-|>',
                          connectionstyle='arc3,rad=0.2',
                          min_target_margin=30,
                          min_source_margin=20)

# Draw nodes with category-specific colors and effects
for category, category_nodes in node_categories.items():
    nx.draw_networkx_nodes(dag, pos,
                          nodelist=category_nodes,
                          node_size=4000,
                          node_color=node_colors[category],
                          edgecolors='#4682B4',
                          linewidths=2,
                          alpha=0.9)

# Add labels with improved styling
labels = {node: node for node in nodes}
nx.draw_networkx_labels(dag, pos,
                       labels,
                       font_size=11,
                       font_weight='bold',
                       font_family='sans-serif',
                       bbox=dict(facecolor='white',
                               edgecolor='#4682B4',
                               boxstyle='round,pad=0.5',
                               alpha=0.8))

# Add title and legend
plt.title("Figure 1. Causal Pathways in Multimorbidity",
         fontsize=18,
         fontweight='bold',
         fontfamily='sans-serif',
         pad=20)

# Add legend with improved positioning
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                             markerfacecolor=color, markersize=15,
                             label=category.replace('_', ' ').title())
                  for category, color in node_colors.items()]

legend_elements.append(plt.Line2D([0], [0], color='#4682B4', 
                                 marker='>', markersize=15,
                                 label='Causal Direction'))

plt.legend(handles=legend_elements,
          loc='upper left',
          bbox_to_anchor=(1, 1),
          fontsize=12,
          title='Legend',
          title_fontsize=14)

# Final styling
plt.margins(0.2)
plt.tight_layout()
plt.axis('off')
plt.show()

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Create DataFrame from the data
data = pd.DataFrame([
    ['E785-I10', 170447, 181586, 17595, 143666, 54365],
    ['E785-Z79899', 165776, 163316, 15897, 189473, 70620],
    ['E785-Z87891', 145589, 148628, 12470, 125475, 46286],
    ['E119-I10', 125735, 119386, 11421, 116090, 42607],
    ['E785-Z7982', 123493, 108016, 8877, 97337, 34117],
    ['E785-K219', 117171, 119840, 9228, 106601, 43127],
    ['D509-D631', 115542, 118058, 7667, 101676, 18379],
    ['E119-E785', 107198, 96972, 8757, 94424, 33493],
    ['F419-F329', 88253, 71550, 1269, 8911, 2685]
], columns=['ICD_Pair', '2020', '2021', '2022', '2023', '2024'])

# Create a mapping of ICD codes to conditions
icd_mapping = {
    'E785': 'Hyperlipidemia',
    'I10': 'Hypertension',
    'Z79899': 'Other Drug Therapy',
    'Z87891': 'History of Nicotine Dependence',
    'E119': 'Type 2 Diabetes',
    'Z7982': 'Long-term Drug Therapy',
    'K219': 'GERD',
    'D509': 'Iron Deficiency Anemia',
    'D631': 'Anemia in Chronic Disease',
    'F419': 'Anxiety',
    'F329': 'Depression'
}

# Create directed graph
G = nx.DiGraph()

# Add edges based on temporal patterns and frequency
for _, row in data.iterrows():
    codes = row['ICD_Pair'].split('-')
    if len(codes) == 2:
        code1, code2 = codes
        # Calculate average co-occurrence
        avg_occurrence = np.mean([row['2020'], row['2021'], row['2022'], row['2023'], row['2024']])
        
        # Add edge with weight based on average occurrence
        if code1 in icd_mapping and code2 in icd_mapping:
            G.add_edge(icd_mapping[code1], icd_mapping[code2], weight=avg_occurrence)

# Draw the network
plt.figure(figsize=(15, 10), facecolor='white')
pos = nx.spring_layout(G, k=2, iterations=50)

# Draw edges with width proportional to weight
edge_weights = [G[u][v]['weight']/5000 for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, width=edge_weights, edge_color='#4682B4', 
                      arrowsize=20, arrowstyle='-|>')

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=3000, node_color='lightblue',
                      edgecolors='#4682B4', linewidths=2)

# Add labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold',
                       bbox=dict(facecolor='white', edgecolor='#4682B4',
                               boxstyle='round,pad=0.5', alpha=0.8))

plt.title("Disease Relationship Network\nBased on 2020-2024 Co-occurrence Patterns",
          fontsize=16, pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

# Calculate temporal correlations
def calculate_temporal_patterns(data):
    patterns = []
    for _, row in data.iterrows():
        codes = row['ICD_Pair'].split('-')
        if len(codes) == 2:
            code1, code2 = codes
            if code1 in icd_mapping and code2 in icd_mapping:
                condition1 = icd_mapping[code1]
                condition2 = icd_mapping[code2]
                years_data = row[['2020', '2021', '2022', '2023', '2024']].values
                
                # Calculate year-over-year changes
                changes = np.diff(years_data)
                consistency = np.mean(np.sign(changes) == np.sign(changes[0]))
                
                patterns.append({
                    'Primary': condition1,
                    'Secondary': condition2,
                    'Avg_Occurrence': np.mean(years_data),
                    'Pattern_Consistency': consistency
                })
    
    return pd.DataFrame(patterns)

temporal_patterns = calculate_temporal_patterns(data)
print("\nTemporal Pattern Analysis:")
print(temporal_patterns.sort_values('Avg_Occurrence', ascending=False))

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Define the directed acyclic graph (DAG) structure
dag = nx.DiGraph()

# Nodes
nodes = [
    "Hypertension", 
    "Hyperlipidemia", 
    "GERD", 
    "Diabetes", 
    "Chronic Kidney Disease",
    "Asthma", 
    "COPD", 
    "Pneumonia", 
    "Acute Respiratory Failure", 
    "Systemic Inflammation"
]

# Edges representing causal pathways
edges = [
    ("Hypertension", "COPD"),
    ("Hyperlipidemia", "COPD"),
    ("Hyperlipidemia", "Acute Respiratory Failure"),
    ("GERD", "Asthma"),
    ("Diabetes", "Chronic Kidney Disease"),
    ("Chronic Kidney Disease", "Acute Respiratory Failure"),
    ("COPD", "Pneumonia"),
    ("Pneumonia", "Acute Respiratory Failure"),
    ("Systemic Inflammation", "COPD"),
    ("Systemic Inflammation", "Asthma"),
    ("Systemic Inflammation", "Acute Respiratory Failure")
]

# Add nodes and edges to the graph
dag.add_nodes_from(nodes)
dag.add_edges_from(edges)

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(dag, seed=42, k=1)  # Increased k for more spacing

# Draw nodes
nx.draw_networkx_nodes(dag, pos, 
                      node_size=3000, 
                      node_color="lightblue", 
                      edgecolors="black",
                      linewidths=2)

# Draw edges with enhanced arrows
nx.draw_networkx_edges(dag, pos,
                      arrowsize=80,  # Increased arrow size
                      arrowstyle='->',  # Changed arrow style
                      edge_color="black",
                      width=2,
                      connectionstyle='arc3,rad=0.2')  # Add curve to edges

# Draw labels with white background for better readability
labels = {node: node for node in nodes}
nx.draw_networkx_labels(dag, pos, 
                       labels,
                       font_size=10,
                       font_weight='bold',
                       bbox=dict(facecolor='white', 
                                edgecolor='none', 
                                alpha=0.7,
                                pad=2))

plt.title("Figure 1. Causal Pathways in Multimorbidity", 
         fontsize=16,
         pad=20)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Function to count ICD code pairs for a specific year
def count_icd_pairs_by_year(data, year):
    icd_data_year = data[data['FROM_YEAR'] == year][icd_diag_columns]
    all_pairs = icd_data_year.apply(extract_pairs, axis=1).explode().dropna()
    pair_counts = Counter(all_pairs)
    return pd.DataFrame(pair_counts.items(), columns=["ICD_Code_Pair", f"Count_{year}"])

# Process data for all years (2020–2024)
years = range(2020, 2025)
yearly_counts = []

for year in years:
    yearly_counts.append(count_icd_pairs_by_year(df, year))

# Merge all yearly data into a single DataFrame
merged_pairs = yearly_counts[0]

for i in range(1, len(yearly_counts)):
    merged_pairs = pd.merge(
        merged_pairs, yearly_counts[i], on="ICD_Code_Pair", how="outer"
    ).fillna(0)

# Calculate temporal changes
for year in years[1:]:
    prev_year = f"Count_{year - 1}"
    curr_year = f"Count_{year}"
    merged_pairs[f"Change_{prev_year}_to_{curr_year}"] = (
        merged_pairs[curr_year] - merged_pairs[prev_year]
    )

# Save the longitudinal analysis to a CSV
merged_pairs.to_csv("icd_code_pairs_longitudinal_2020_2024.csv", index=False)

# Display some results
print("Top temporal changes in ICD code pairs:")
print(merged_pairs.sort_values(by=f"Change_{years[-2]}_to_{years[-1]}", ascending=False).head(10))


In [ ]:
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Function to count ICD code pairs for a specific year
def count_icd_pairs_by_year(data, year):
    icd_data_year = data[data['FROM_YEAR'] == year][icd_diag_columns]
    all_pairs = icd_data_year.apply(extract_pairs, axis=1).explode().dropna()
    pair_counts = Counter(all_pairs)
    return pd.DataFrame(pair_counts.items(), columns=["ICD_Code_Pair", f"Count_{year}"])

# Count pairs for each year
pairs_2023 = count_icd_pairs_by_year(df, 2023)
pairs_2024 = count_icd_pairs_by_year(df, 2024)

# Merge the two datasets for comparison
merged_pairs = pd.merge(
    pairs_2023, pairs_2024, on="ICD_Code_Pair", how="outer"
).fillna(0)

# Add a column for the difference in counts between the years
merged_pairs['Difference'] = merged_pairs['Count_2024'] - merged_pairs['Count_2023']

# Save the results to a CSV
merged_pairs.to_csv("icd_code_pairs_longitudinal.csv", index=False)

# Display top changes
print("Top changes in ICD code pairs:")
print(merged_pairs.sort_values(by="Difference", ascending=False).head(10))


In [ ]:
# cover 2020 - 2024 
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Function to count ICD code pairs for a specific year
def count_icd_pairs_by_year(data, year):
    icd_data_year = data[data['FROM_YEAR'] == year][icd_diag_columns]
    all_pairs = icd_data_year.apply(extract_pairs, axis=1).explode().dropna()
    pair_counts = Counter(all_pairs)
    return pd.DataFrame(pair_counts.items(), columns=["ICD_Code_Pair", f"Count_{year}"])

# Process data for all years (2020–2024)
years = range(2020, 2025)
yearly_counts = []

for year in years:
    yearly_counts.append(count_icd_pairs_by_year(df, year))

# Merge all yearly data into a single DataFrame
merged_pairs = yearly_counts[0]

for i in range(1, len(yearly_counts)):
    merged_pairs = pd.merge(
        merged_pairs, yearly_counts[i], on="ICD_Code_Pair", how="outer"
    ).fillna(0)

# Calculate temporal changes
for year in years[1:]:
    prev_year = f"Count_{year - 1}"
    curr_year = f"Count_{year}"
    merged_pairs[f"Change_{prev_year}_to_{curr_year}"] = (
        merged_pairs[curr_year] - merged_pairs[prev_year]
    )

# Save the longitudinal analysis to a CSV
merged_pairs.to_csv("icd_code_pairs_longitudinal_2020_2024.csv", index=False)

# Display some results
print("Top temporal changes in ICD code pairs:")
print(merged_pairs.sort_values(by=f"Change_{years[-2]}_to_{years[-1]}", ascending=False).head(10))


In [ ]:
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Filter only the ICD diagnosis columns from the DataFrame
icd_data = df[icd_diag_columns]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Flatten all pairs across rows
all_pairs = icd_data.apply(extract_pairs, axis=1).explode().dropna()

# Count the frequency of each pair
pair_counts = Counter(all_pairs)

# Get the top 10 most common pairs
top_10_pairs = pair_counts.most_common(1000)

# Convert to DataFrame for easier viewing
top_10_pairs_df = pd.DataFrame(top_10_pairs, columns=["ICD_Code_Pair", "Count"])

# Display the results
print("Top 10 ICD code pairs:")
print(top_10_pairs_df)

# Save the results to a CSV
top_10_pairs_df.to_csv("top_10_icd_code_pairs.csv", index=False)


In [ ]:
# Get the top 10 most common pairs
top_10_pairs = pair_counts.most_common(1000)

# Convert to DataFrame for easier viewing
top_10_pairs_df = pd.DataFrame(top_10_pairs, columns=["ICD_Code_Pair", "Count"])

# Display the results
print("Top 10 ICD code pairs:")
print(top_10_pairs_df)

# Save the results to a CSV
top_10_pairs_df.to_csv("top_10_icd_code_pairs.csv", index=False)


In [ ]:
pair_counts